# Ders 6: ARIMA ve Mevsimsel ARIMA (SARIMA) Modelleri
Bu notebook'ta şunları öğreneceğiz:
- Entegre süreçler ve farklama (ARIMA)
- Birim kök testleri (ADF, KPSS)
- Mevsimsel ARIMA (SARIMA) modelleri
- Gerçek veri üzerinde tam modelleme iş akışı


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Kütüphaneler yüklendi.")


## 6.1 Entegre Süreçler ve Farklama

Bir $\{X_t\}$ süreci **ARIMA(p,d,q)** sürecidir; eğer $d$. dereceden farklama sonucunda elde edilen
$$Y_t = (1-B)^d X_t$$
süreci **ARMA(p,q)** süreciyse.

### Farklama Operatörü
- **Birinci fark:** $(1-B)X_t = X_t - X_{t-1}$
- **İkinci fark:** $(1-B)^2 X_t = X_t - 2X_{t-1} + X_{t-2}$
- **Mevsimsel fark:** $(1-B^s)X_t = X_t - X_{t-s}$

**Ne zaman farklama?** Seri durağan değilse (trend veya birim kök varsa).


In [ ]:
# ── Farklama Etkisinin Görselleştirilmesi ──

np.random.seed(42)
n = 200

# ARIMA(1,1,0) — birinci farklama gerektiren seri
# Gerçek süreç: Y_t = Y_{t-1} + epsilon_t (rastgele yürüyüş)
epsilon = np.random.normal(0, 1, n)
X = np.cumsum(epsilon)  # Rastgele yürüyüş = entegre WN

# 1. ve 2. farklar
dX = np.diff(X, n=1)   # (1-B)X_t
ddX = np.diff(X, n=2)  # (1-B)^2 X_t

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(X, color='steelblue', lw=1.2)
axes[0].set_title('Orijinal Seri $X_t$ — Rastgele Yürüyüş (Durağan Değil)', fontweight='bold')
axes[0].axhline(X.mean(), color='red', ls='--', alpha=0.5, label=f'Ortalama={X.mean():.1f}')
axes[0].legend()

axes[1].plot(dX, color='darkorange', lw=1.2)
axes[1].set_title('Birinci Fark $(1-B)X_t$ — Durağan ✓', fontweight='bold')
axes[1].axhline(0, color='red', ls='--', alpha=0.5)

axes[2].plot(ddX, color='green', lw=1.2)
axes[2].set_title('İkinci Fark $(1-B)^2 X_t$ — Aşırı Farklama', fontweight='bold')
axes[2].axhline(0, color='red', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Orijinal seri varyansı:       {np.var(X):.2f}")
print(f"Birinci fark varyansı:        {np.var(dX):.2f}")
print(f"İkinci fark varyansı:         {np.var(ddX):.2f}")
print("\nNot: Aşırı farklama varyansı artırır!")


## 6.2 Birim Kök Testleri

### Augmented Dickey-Fuller (ADF) Testi
$$H_0: \phi = 1 \text{ (birim kök var, durağan değil)}$$
$$H_1: |\phi| < 1 \text{ (durağan)}$$

**Karar:** p-değeri < 0.05 ise $H_0$ reddedilir → seri durağan.

### KPSS Testi
$$H_0: \text{Seri durağan}$$
$$H_1: \text{Birim kök var}$$

**Not:** ADF ve KPSS birlikte kullanılmalıdır — sonuçlar çelişebilir.


In [ ]:
# ── ADF ve KPSS Testleri ──

def birim_kok_testleri(seri, ad="Seri"):
    print(f"{'='*55}")
    print(f"  {ad}")
    print(f"{'='*55}")
    
    # ADF Testi
    adf_result = adfuller(seri, autolag='AIC')
    print(f"\nADF Testi:")
    print(f"  İstatistik : {adf_result[0]:.4f}")
    print(f"  p-değeri   : {adf_result[1]:.4f}")
    print(f"  Kritik     : 1%={adf_result[4]['1%']:.3f}, 5%={adf_result[4]['5%']:.3f}")
    if adf_result[1] < 0.05:
        print(f"  Karar      : Durağan (H0 reddedildi)")
    else:
        print(f"  Karar      : Durağan DEĞİL (H0 reddedilemedi)")
    
    # KPSS Testi
    kpss_result = kpss(seri, regression='c', nlags='auto')
    print(f"\nKPSS Testi:")
    print(f"  İstatistik : {kpss_result[0]:.4f}")
    print(f"  p-değeri   : {kpss_result[1]:.4f}")
    print(f"  Kritik     : 5%={kpss_result[3]['5%']:.3f}")
    if kpss_result[1] > 0.05:
        print(f"  Karar      : Durağan (H0 reddedilemedi)")
    else:
        print(f"  Karar      : Durağan DEĞİL (H0 reddedildi)")

np.random.seed(42)
n = 200
eps = np.random.normal(0, 1, n)

# 3 farklı seri
rw = np.cumsum(eps)                         # Rastgele yürüyüş
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.8*ar1[t-1] + eps[t]         # AR(1) durağan

birim_kok_testleri(rw, "Rastgele Yürüyüş (Birim Kök Beklenir)")
birim_kok_testleri(ar1, "AR(1) phi=0.8 (Durağan Beklenir)")
birim_kok_testleri(np.diff(rw), "Farklananmış RW (Durağan Beklenir)")


## 6.3 ARIMA(p,d,q) Modeli Seçimi

**Pratik Kural:**
1. ACF/PACF grafiklerini çiz
2. Seri durağan değilse farklama uygula ($d$ = farklama sayısı)
3. Farklanmış serinin ACF/PACF'ine bakarak $p$ ve $q$ belirle
4. AICC ile model seç

| Model | ACF | PACF |
|-------|-----|------|
| AR(p) | Yavaş azalır | $p$ lag sonra kesilir |
| MA(q) | $q$ lag sonra kesilir | Yavaş azalır |
| ARMA | Yavaş azalır | Yavaş azalır |


In [ ]:
# ── ARIMA Model Uydurma: AirPassengers ──

from statsmodels.datasets import get_rdataset

# Havayolu yolcu verisi
try:
    data = get_rdataset('AirPassengers')
    ap = data.data['value'].values
except:
    # Elle oluştur (1949-1960 aylık veri)
    ap = np.array([112,118,132,129,121,135,148,148,136,119,104,118,
                   115,126,141,135,125,149,170,170,158,133,114,140,
                   145,150,178,163,172,178,199,199,184,162,146,166,
                   171,180,193,181,183,218,230,242,209,191,172,194,
                   196,196,236,235,229,243,264,272,237,211,180,201,
                   204,188,235,227,234,264,302,293,259,229,203,229,
                   242,233,267,269,270,315,364,347,312,274,237,278,
                   284,277,317,313,318,374,413,405,355,306,271,306,
                   315,301,356,348,355,422,465,467,404,347,305,336,
                   340,318,362,348,363,435,491,505,404,359,310,337,
                   360,342,406,396,420,472,548,559,463,407,362,405,
                   417,391,419,461,472,535,622,606,508,461,390,432])

t_idx = pd.date_range('1949-01', periods=len(ap), freq='MS')
ap_series = pd.Series(ap, index=t_idx)

# Log dönüşümü (varyans stabilizasyonu)
log_ap = np.log(ap_series)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(ap_series, color='steelblue')
axes[0,0].set_title('Orijinal Seri', fontweight='bold')

axes[0,1].plot(log_ap, color='darkorange')
axes[0,1].set_title('Log Dönüşümü', fontweight='bold')

# Mevsimsel fark + birinci fark
diff_log = log_ap.diff(12).diff(1).dropna()
axes[1,0].plot(diff_log, color='green')
axes[1,0].set_title('Log + Mevsimsel Fark (s=12) + Birinci Fark', fontweight='bold')
axes[1,0].axhline(0, color='red', ls='--', alpha=0.5)

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(diff_log, lags=30, ax=axes[1,1], title='ACF — Farklanmış Log Seri')

plt.tight_layout()
plt.show()

# ADF testi farklanmış seriye
adf = adfuller(diff_log, autolag='AIC')
print(f"ADF p-değeri (farklanmış): {adf[1]:.4f} {'→ Durağan ✓' if adf[1]<0.05 else '→ Durağan değil'}")


## 6.4 SARIMA$(p,d,q)(P,D,Q)_s$ Modelleri

Mevsimsel ARIMA modeli hem mevsimsel hem mevsimsel olmayan bileşenler içerir:

$$\Phi_P(B^s)\phi_p(B)(1-B^s)^D(1-B)^d X_t = \Theta_Q(B^s)\theta_q(B)\epsilon_t$$

**Parametre seçimi:**
- $s$ = mevsim periyodu (aylık için $s=12$, çeyreklik için $s=4$)
- $(p,d,q)$ = mevsimsel olmayan bileşen
- $(P,D,Q)$ = mevsimsel bileşen


In [ ]:
# ── SARIMA(1,1,1)(1,1,1)_12 — AirPassengers ──

from statsmodels.tsa.statespace.sarimax import SARIMAX

# SARIMA modelini fit et
model = SARIMAX(log_ap,
                order=(1,1,1),
                seasonal_order=(1,1,1,12),
                trend='n')
fit = model.fit(disp=False)
print(fit.summary().tables[0])
print(fit.summary().tables[1])

# Artık diagnostik
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
residuals = fit.resid

axes[0,0].plot(residuals, color='steelblue', lw=0.8)
axes[0,0].set_title('Artıklar', fontweight='bold')
axes[0,0].axhline(0, color='red', ls='--')

from scipy import stats
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('QQ Grafiği', fontweight='bold')

plot_acf(residuals, lags=30, ax=axes[1,0], title='Artık ACF')
plot_pacf(residuals, lags=30, ax=axes[1,1], title='Artık PACF')

plt.tight_layout()
plt.show()

# AIC
print(f"\nAIC: {fit.aic:.2f}  |  BIC: {fit.bic:.2f}")


In [ ]:
# ── SARIMA ile 24 Aylık Öngörü ──

forecast_steps = 24
pred = fit.get_forecast(steps=forecast_steps)
pred_mean = pred.predicted_mean
pred_ci = pred.conf_int(alpha=0.05)

# Log ölçekten orijinal ölçeğe dön
future_idx = pd.date_range(ap_series.index[-1] + pd.DateOffset(months=1),
                            periods=forecast_steps, freq='MS')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ap_series, label='Gözlem', color='steelblue')
ax.plot(future_idx, np.exp(pred_mean), label='Öngörü', color='red', lw=2)
ax.fill_between(future_idx,
                np.exp(pred_ci.iloc[:,0]),
                np.exp(pred_ci.iloc[:,1]),
                color='red', alpha=0.2, label='%95 GA')
ax.set_title('SARIMA(1,1,1)(1,1,1)₁₂ — AirPassengers 24 Aylık Öngörü',
             fontweight='bold', fontsize=13)
ax.legend()
ax.set_xlabel('Tarih')
ax.set_ylabel('Yolcu Sayısı (bin)')
plt.tight_layout()
plt.show()


## Özet

| Kavram | Açıklama |
|--------|----------|
| **ARIMA(p,d,q)** | $d$ kez farklanmış ARMA(p,q) |
| **Birim kök** | ADF (H₀: birim kök var), KPSS (H₀: durağan) |
| **SARIMA(p,d,q)(P,D,Q)ₛ** | Mevsimsel bileşenli ARIMA |
| **Mevsimsel fark** | $(1-B^s)X_t = X_t - X_{t-s}$ |
| **Log dönüşümü** | Varyans stabilizasyonu için |

**Modelleme Adımları:**
1. Seriyi görselleştir → trend/mevsimsellik var mı?
2. Log dönüşümü (gerekiyorsa)
3. Birim kök testi → kaç farklama?
4. ACF/PACF → p, q, P, Q seç
5. SARIMAX ile fit et
6. Artık diagnostik → model uygun mu?
7. Öngörü üret
